# WESPER: fine-tune the encoder for Swedish whispers

This notebook fine-tunes WESPER's speech-to-unit encoder so that **Swedish whispers produce the
same speech units as Swedish normal speech**.

- **Data:** validated Common Voice sv-SE clips, each paired with its Normal2Whisper pseudo-whisper.
- **How:** a frozen copy of the original encoder ("teacher") turns each *normal* clip into speech
  units. The encoder being trained ("student") gets the *whispered* version (sometimes the normal
  one, so normal speech keeps working) and learns to produce the teacher's units.
- **Why this way:** the units stay in the original encoder's format, so WESPER's existing decoder
  and vocoder keep working. You can listen to results at the end, and use the new encoder with
  `convert.py --hubert`. The output voice is still WESPER's English decoder voice.

**Before you start**
1. Upload `wesper-sv-n2w.tar` to your Google Drive, into a folder `wesper-sv` (so it's at
   `MyDrive/wesper-sv/wesper-sv-n2w.tar`).
2. Pick a GPU runtime: *Runtime → Change runtime type → GPU*.
3. *Runtime → Run all*. Allow Drive access when asked.

Checkpoints are saved to `MyDrive/wesper-sv/runs/<RUN_NAME>/`. If Colab disconnects, run all
again: training resumes from the last checkpoint.

In [ ]:
# @title Settings
# Where the data tar lives and where checkpoints go (on your Google Drive).
DRIVE_DIR = "/content/drive/MyDrive/wesper-sv"  # @param {type:"string"}
DATA_TAR = "wesper-sv-n2w.tar"  # @param {type:"string"}
RUN_NAME = "n2w-finetune"  # @param {type:"string"}

# Training length and batch. One step sees BATCH_SIZE crops of CROP_SECONDS each.
MAX_STEPS = 10000  # @param {type:"integer"}
BATCH_SIZE = 16  # @param {type:"integer"}
CROP_SECONDS = 3.0  # @param {type:"number"}
LEARNING_RATE = 3e-5  # @param {type:"number"}
WARMUP_STEPS = 500  # @param {type:"integer"}

# Share of training inputs that are whispered; the rest are normal speech.
WHISPER_PROB = 0.75  # @param {type:"number"}
# Random +/- gain on training inputs, for robustness to small level errors.
GAIN_JITTER_DB = 3.0  # @param {type:"number"}

EVAL_EVERY = 500  # @param {type:"integer"}
SAVE_EVERY = 1000  # @param {type:"integer"}
VAL_CLIPS = 200  # @param {type:"integer"}
LISTEN_CLIPS = 4  # @param {type:"integer"}

LOCAL_DATA_DIR = "/content/data"
WESPER_DIR = "/content/wesper-demo"
WESPER_REPO = "https://github.com/rkmt/wesper-demo.git"
WESPER_COMMIT = "25a5b344ac33fb6a55e28f0be6a4b145500e74b5"
RELEASE = "https://github.com/rkmt/wesper-demo/releases/download/v0.1"
ENCODER_URL = f"{RELEASE}/model-layer12-450000.pt"
NUM_WORKERS = 2
SEED = 0

## Setup
Mounts Drive, fetches the WESPER code, and checks the GPU.

In [ ]:
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    # Needed by WESPER's FastSpeech2 text module, which the decoder imports.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "inflect", "unidecode"], check=True)

if not os.path.exists(WESPER_DIR):
    subprocess.run(["git", "clone", "-q", WESPER_REPO, WESPER_DIR], check=True)
    subprocess.run(["git", "-C", WESPER_DIR, "checkout", "-q", WESPER_COMMIT], check=True)
# WESPER's code loads configs by paths relative to its repo root.
os.chdir(WESPER_DIR)
if WESPER_DIR not in sys.path:
    sys.path.insert(0, WESPER_DIR)

import contextlib
import random

import numpy as np
import torch

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    # bfloat16 on Ampere or newer (A100, L4); float16 with loss scaling on older GPUs (T4).
    AMP_DTYPE = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
    print(f"GPU: {torch.cuda.get_device_name(0)}, mixed precision: {AMP_DTYPE}")
else:
    AMP_DTYPE = None
    print("No GPU found: training will be very slow. In Colab: Runtime > Change runtime type > GPU.")


def autocast():
    return torch.autocast("cuda", dtype=AMP_DTYPE) if AMP_DTYPE else contextlib.nullcontext()


RUN_DIR = os.path.join(DRIVE_DIR, "runs", RUN_NAME)
os.makedirs(RUN_DIR, exist_ok=True)
print("Checkpoints:", RUN_DIR)

## Data
Copies the tar from Drive to the Colab disk (much faster to read than Drive) and unpacks it.
`train` is every validated clip from speakers outside Common Voice's test set; `val` is clips from
test-set speakers, so validation measures voices the encoder never trained on.

The clips were already processed by `prepare_data.py`: a touchpad thud at the start is muted, and
every clip is normalized to the same speech level. Common Voice levels vary by 30+ dB between
clips, and the encoder is very level-sensitive (−10 dB changes its output by about 30%). The same
normalization is applied when converting your own recordings below.

In [ ]:
import csv
import json
import shutil
import time
from collections import Counter

done_marker = os.path.join(LOCAL_DATA_DIR, ".extracted")
if not os.path.exists(done_marker):
    os.makedirs(LOCAL_DATA_DIR, exist_ok=True)
    started = time.time()
    local_tar = os.path.join(LOCAL_DATA_DIR, DATA_TAR)
    shutil.copy(os.path.join(DRIVE_DIR, DATA_TAR), local_tar)
    subprocess.run(["tar", "-xf", local_tar, "-C", LOCAL_DATA_DIR], check=True)
    os.remove(local_tar)
    open(done_marker, "w").close()
    print(f"Copied and unpacked in {time.time() - started:.0f} s")

with open(os.path.join(LOCAL_DATA_DIR, "prep.json")) as f:
    PREP = json.load(f)
TARGET_DBFS, MAX_GAIN_DB = PREP["target_dbfs"], PREP["max_gain_db"]
# Clips are stored HEADROOM_DB below the training level so 16-bit FLAC doesn't clip.
HEADROOM = 10 ** (PREP["headroom_db"] / 20)

with open(os.path.join(LOCAL_DATA_DIR, "manifest.tsv"), newline="") as f:
    manifest = list(csv.DictReader(f, delimiter="\t", quoting=csv.QUOTE_NONE))
levels = np.array([float(r["normal_dbfs"]) for r in manifest])
thuds = np.array([float(r["thud_ms"]) for r in manifest])
print(f"Speech level before normalization: {np.percentile(levels, 5):.1f} to {np.percentile(levels, 95):.1f} dBFS "
      f"(5th-95th percentile), now all {TARGET_DBFS} dBFS")
print(f"Thud muted at the start of {(thuds > 0).mean() * 100:.0f}% of clips")
train_rows = [r for r in manifest if r["split"] == "train"]
val_rows = [r for r in manifest if r["split"] == "val"][:VAL_CLIPS]
for name, rows in (("train", train_rows), ("val", val_rows)):
    hours = sum(int(r["duration_ms"]) for r in rows) / 3.6e6
    print(f"{name}: {len(rows)} clips, {hours:.1f} h, {len({r['speaker'] for r in rows})} speakers")

In [ ]:
import soundfile as sf
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler

SR = 16000
HOP = 320  # the encoder outputs one unit vector per 320 samples (20 ms)
CROP = int(CROP_SECONDS * SR) // HOP * HOP


def load_pair(clip):
    # The processed normal and whispered versions, same length, at TARGET_DBFS speech level.
    normal, _ = sf.read(f"{LOCAL_DATA_DIR}/normal/{clip}.flac", dtype="float32")
    whisper, _ = sf.read(f"{LOCAL_DATA_DIR}/whisper/{clip}.flac", dtype="float32")
    return normal * HEADROOM, whisper * HEADROOM


def speech_dbfs(x):
    # Speech level in dBFS, ignoring silence, background noise and clicks. Same as speech_dbfs()
    # in prepare_data.py, which normalized the training data with it.
    n = len(x) // HOP
    if n == 0:
        return -120.0
    energy = np.mean(x[: n * HOP].astype(np.float64).reshape(n, HOP) ** 2, axis=1)
    speech = energy[energy >= np.percentile(energy, 99) / 1000]  # drop frames 30 dB below the loudest
    ref = np.percentile(speech, 90)
    active = np.minimum(speech[speech >= ref / 100], ref)  # cap frames so a click can't dominate
    return float(10 * np.log10(active.mean() + 1e-12))


def normalize(x):
    # Scale new audio (e.g. your own recording) to TARGET_DBFS speech level, like the training
    # data. It stays float, so peaks above 1.0 are fine: nothing clips on the way into the encoder.
    gain = min(TARGET_DBFS - speech_dbfs(x), MAX_GAIN_DB)
    return (x * 10 ** (gain / 20)).astype(np.float32)


class PairDataset(Dataset):
    # Yields (input, target, frames): input is the whispered or normal crop, target the normal crop.

    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        normal, whisper = load_pair(self.rows[i]["clip"])
        use_whisper = random.random() < WHISPER_PROB
        jitter = 10 ** (random.uniform(-GAIN_JITTER_DB, GAIN_JITTER_DB) / 20)
        target, source = normal, (whisper if use_whisper else normal) * jitter
        if len(target) > CROP:
            # The two versions are time-aligned, so the same window is the same speech.
            start = random.randrange(len(target) - CROP + 1)
            target, source = target[start:start + CROP], source[start:start + CROP]
        frames = len(target) // HOP
        pad = CROP - len(target)
        return torch.from_numpy(np.pad(source, (0, pad))), torch.from_numpy(np.pad(target, (0, pad))), frames


# Two speakers recorded about half of the training clips. Sampling with weight 1/sqrt(clips per
# speaker) evens that out somewhat without discarding their data.
per_speaker = Counter(r["speaker"] for r in train_rows)
weights = [1 / np.sqrt(per_speaker[r["speaker"]]) for r in train_rows]
loader = DataLoader(
    PairDataset(train_rows),
    batch_size=BATCH_SIZE,
    sampler=WeightedRandomSampler(weights, num_samples=len(train_rows), replacement=True),
    num_workers=NUM_WORKERS,
    pin_memory=DEVICE == "cuda",
    drop_last=True,
    persistent_workers=NUM_WORKERS > 0,
)

## Encoders
Downloads the original WESPER encoder (1.1 GB). The teacher is a frozen copy of it; the student
starts from the same weights and is the one being trained.

In [ ]:
import torch.nn.functional as F
from torch.nn.modules.utils import consume_prefix_in_state_dict_if_present

from libs.hubert.model import HubertSoft


def load_encoder(path_or_url):
    if path_or_url.startswith("http"):
        cached = os.path.join(torch.hub.get_dir(), "checkpoints", os.path.basename(path_or_url))
        if not os.path.exists(cached):
            os.makedirs(os.path.dirname(cached), exist_ok=True)
            torch.hub.download_url_to_file(path_or_url, cached)
        path_or_url = cached
    state = torch.load(path_or_url, map_location="cpu", weights_only=False)["hubert"]
    consume_prefix_in_state_dict_if_present(state, "module.")
    encoder = HubertSoft()
    encoder.load_state_dict(state, strict=True)
    return encoder


def soft_units(encoder, wav):
    # Same as HubertSoft.units(), minus its inference_mode so gradients can flow.
    # wav: [batch, samples] -> units: [batch, samples // 320, 256]
    wav = F.pad(wav.unsqueeze(1), ((400 - 320) // 2, (400 - 320) // 2))
    x, _ = encoder.encode(wav)
    return encoder.proj(x)


student = load_encoder(ENCODER_URL).to(DEVICE)
student._mask = False  # turn off HuBERT's training-time masking; we want a plain mapping
# A second copy from the same file: copy.deepcopy fails on the encoder's weight_norm layer.
teacher = load_encoder(ENCODER_URL).to(DEVICE).eval().requires_grad_(False)
print(f"Encoder: {sum(p.numel() for p in student.parameters()) / 1e6:.1f} M parameters")

In [ ]:
# Validation: how far the units from whispered (and normal) input are from the teacher's units
# for the matching normal clip. Lower is better; "normal" should stay near zero.
val_items = []
for r in val_rows:
    normal, whisper = load_pair(r["clip"])
    n = len(normal) // HOP * HOP
    normal, whisper = normal[:n], whisper[:n]
    with torch.no_grad(), autocast():
        target = soft_units(teacher, torch.from_numpy(normal)[None].to(DEVICE))[0].float().cpu()
    val_items.append((normal, whisper, target))


@torch.no_grad()
def evaluate(encoder):
    was_training = encoder.training
    encoder.eval()
    errors = {"whisper": [], "normal": []}
    for normal, whisper, target in val_items:
        for kind, wav in (("whisper", whisper), ("normal", normal)):
            with autocast():
                units = soft_units(encoder, torch.from_numpy(wav)[None].to(DEVICE))[0].float().cpu()
            errors[kind].append((units - target).abs().mean().item())
    encoder.train(was_training)
    return {kind: float(np.mean(e)) for kind, e in errors.items()}


baseline = evaluate(teacher)
print(f"Original encoder, distance to normal-speech units: whisper {baseline['whisper']:.4f}, normal {baseline['normal']:.4f}")

## Train
Prints progress every 100 steps and validates every `EVAL_EVERY` steps. `encoder_best.pt` (the
best validation score so far) and `latest.pt` (everything needed to resume) are saved to Drive.
You can stop the cell at any time; run it again to continue.

In [ ]:
import json
import math

optimizer = torch.optim.AdamW(student.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
# Loss scaling is only needed for float16 (T4); bfloat16 and float32 don't use it.
GradScaler = torch.amp.GradScaler if hasattr(torch.amp, "GradScaler") else torch.cuda.amp.GradScaler
scaler = GradScaler(enabled=AMP_DTYPE == torch.float16)

CONFIG = {k: v for k, v in globals().items() if k.isupper() and isinstance(v, (int, float, str))}
latest_path = os.path.join(RUN_DIR, "latest.pt")
best_path = os.path.join(RUN_DIR, "encoder_best.pt")
step, best, history = 0, float("inf"), {"train": [], "val": [{"step": 0, **baseline}]}
if os.path.exists(latest_path):
    state = torch.load(latest_path, map_location="cpu", weights_only=False)
    student.load_state_dict(state["hubert"])
    optimizer.load_state_dict(state["optimizer"])
    scaler.load_state_dict(state["scaler"])
    step, best, history = state["step"], state["best"], state["history"]
    print(f"Resuming from step {step}")


def lr_at(step):
    # Linear warmup, then cosine decay to 10% of the peak.
    if step < WARMUP_STEPS:
        return LEARNING_RATE * (step + 1) / WARMUP_STEPS
    progress = min(1.0, (step - WARMUP_STEPS) / max(1, MAX_STEPS - WARMUP_STEPS))
    return LEARNING_RATE * (0.1 + 0.45 * (1 + math.cos(math.pi * progress)))


def save(obj, path):
    # Write then rename, so a disconnect mid-save can't leave a broken checkpoint.
    torch.save(obj, path + ".tmp")
    os.replace(path + ".tmp", path)


def batches():
    while True:
        yield from loader


student.train()
running, started, start_step = [], time.time(), step
for source, target, frames in batches():
    if step >= MAX_STEPS:
        break
    source, target = source.to(DEVICE, non_blocking=True), target.to(DEVICE, non_blocking=True)
    for group in optimizer.param_groups:
        group["lr"] = lr_at(step)

    with autocast():
        with torch.no_grad():
            goal = soft_units(teacher, target)
        units = soft_units(student, source)
    # Mean absolute error over the real (unpadded) frames of each crop.
    mask = (torch.arange(units.shape[1], device=DEVICE)[None, :] < frames.to(DEVICE)[:, None]).unsqueeze(-1)
    loss = ((units.float() - goal.float()).abs() * mask).sum() / (mask.sum() * units.shape[-1])

    optimizer.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
    scaler.step(optimizer)
    scaler.update()
    step += 1
    running.append(loss.item())

    if step % 100 == 0 or step == MAX_STEPS:
        rate = (step - start_step) / (time.time() - started)
        history["train"].append({"step": step, "loss": float(np.mean(running))})
        print(f"step {step}/{MAX_STEPS}  loss {np.mean(running):.4f}  lr {lr_at(step):.1e}  "
              f"{rate:.2f} steps/s  ETA {(MAX_STEPS - step) / rate / 60:.0f} min", flush=True)
        running = []

    if step % EVAL_EVERY == 0 or step == MAX_STEPS:
        scores = evaluate(student)
        history["val"].append({"step": step, **scores})
        improved = scores["whisper"] < best
        if improved:
            best = scores["whisper"]
            save({"hubert": student.state_dict(), "step": step, "val": scores, "config": CONFIG}, best_path)
        print(f"  val: whisper {scores['whisper']:.4f} (original {baseline['whisper']:.4f}), "
              f"normal {scores['normal']:.4f}{'  -> saved encoder_best.pt' if improved else ''}", flush=True)

    if step % SAVE_EVERY == 0 or step == MAX_STEPS:
        save({"hubert": student.state_dict(), "optimizer": optimizer.state_dict(), "scaler": scaler.state_dict(),
              "step": step, "best": best, "history": history, "config": CONFIG}, latest_path)
        with open(os.path.join(RUN_DIR, "history.json"), "w") as f:
            json.dump(history, f, indent=1)

print(f"Done at step {step}. Best validation (whisper): {best:.4f}, original encoder: {baseline['whisper']:.4f}")

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot([h["step"] for h in history["train"]], [h["loss"] for h in history["train"]])
ax1.set(title="Training loss", xlabel="step", ylabel="mean abs error")
for kind in ("whisper", "normal"):
    ax2.plot([h["step"] for h in history["val"]], [h[kind] for h in history["val"]], marker="o", label=f"{kind} input")
ax2.axhline(baseline["whisper"], linestyle="--", color="gray", label="original encoder, whisper input")
ax2.set(title="Validation: distance to normal-speech units", xlabel="step")
ax2.legend()
plt.show()

## Listen
Converts validation clips (speakers the encoder never trained on) with the original and the
fine-tuned encoder, through WESPER's existing decoder and vocoder. For each clip: the normal
recording, the Normal2Whisper input, and the two conversions.

In [ ]:
import argparse

from IPython.display import Audio, display

from whisper_normal import MyWhisper2Normal

w2n = MyWhisper2Normal(argparse.Namespace(
    preprocess_config="config/my_preprocess16k_LJ.yaml",
    model_config="config/my_model16000.yaml",
    device=DEVICE,
    hubert=ENCODER_URL,
    fastspeech2=f"{RELEASE}/googletts_neutral_best.tar",
    hifigan=f"{RELEASE}/g_00205000",
), load_encoder=False)

finetuned = load_encoder(best_path).to(DEVICE).eval() if os.path.exists(best_path) else student.eval()


def convert(wav, encoder):
    w2n.encoder = encoder
    out, _ = w2n.convert(normalize(wav))
    return out.astype(np.float32) / 32768


def show(label, wav):
    print(label)
    display(Audio(wav, rate=SR))


for r in val_rows[:LISTEN_CLIPS]:
    normal, whisper = load_pair(r["clip"])
    print(f"\n{r['sentence']}")
    show("  normal recording", normal)
    show("  Normal2Whisper input", whisper)
    show("  converted, original encoder", convert(whisper, teacher))
    show("  converted, fine-tuned encoder", convert(whisper, finetuned))

### Try your own whisper
Upload a recording of yourself whispering in Swedish (wav, flac, mp3 or m4a). It's converted with
both encoders.

In [ ]:
import tempfile

import librosa

if IN_COLAB:
    from google.colab import files

    for name, data in files.upload().items():
        with tempfile.NamedTemporaryFile(suffix=os.path.splitext(name)[1]) as f:
            f.write(data)
            f.flush()
            wav, _ = librosa.load(f.name, sr=SR, mono=True)
        print(f"\n{name}")
        show("  your recording", wav)
        show("  converted, original encoder", convert(wav, teacher))
        show("  converted, fine-tuned encoder", convert(wav, finetuned))

## Use the new encoder locally
Download `encoder_best.pt` from `MyDrive/wesper-sv/runs/<RUN_NAME>/` and pass it to WESPER:

```sh
python convert.py --hubert encoder_best.pt --input my_whisper.wav --output converted.wav
```

**Normalize the input first.** The encoder is very sensitive to loudness, and it was trained on
audio normalized to `TARGET_DBFS` (default −20 dBFS speech level). `convert.py` doesn't normalize,
so apply the same `normalize()` from this notebook before conversion, as the Listen cells do.